In [1]:
import pandas as pd

df = pd.read_csv('C:/hybrid_filtering/0507_userindex.csv')
mart = pd.read_csv('C:/hybrid_filtering/vod_mart_processed.csv')

C:\Users\user\AppData\Local\Temp\ipykernel_27276\760196564.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  mart = pd.read_csv('C:/hybrid_filtering/vod_mart_processed.csv')


In [2]:
df.drop(columns = {'sha2_hash', 'user_label', 'asset_nm', 'disp_rtm'}, inplace = True)

In [3]:
# mart에서 필요한 컬럼만 선택
mart_subset = mart[['asset_id', 'super_asset_nm', 'category_l2', 'genre', 'disp_rtm', 'smry']]

# df와 mart_subset을 asset_id를 기준으로 병합
df = pd.merge(df, mart_subset, on='asset_id', how='left')

In [4]:
import re

# 괄호와 특수문자를 제거하는 함수 정의
def clean_text(text):
    if pd.isna(text):  # NaN 값 처리
        return text
    # 괄호와 괄호 안의 내용 제거
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    # 특수문자 제거 (한글, 영문, 숫자만 남김)
    text = re.sub(r'[^\w\s가-힣]', '', text)
    # 공백 제거
    text = text.strip()
    return text

# 각 컬럼에 대해 정제 적용
df['super_asset_nm'] = df['super_asset_nm'].apply(clean_text)
df['category_l2'] = df['category_l2'].apply(clean_text)
df['genre'] = df['genre'].apply(clean_text)
df['smry'] = df['smry'].apply(clean_text)

## 파생변수 view_ratio 생성

In [5]:
# use_tms를 disp_rtm으로 나누어 비율 계산
df['disp_rtm'].astype(int)
df['view_ratio'] = (df['use_tms'] / df['disp_rtm']).clip(upper = 1)

In [6]:
df.dropna(subset = 'view_ratio', inplace = True)

### 주기성 반영 파생변수 생성

In [7]:
import numpy as np

# 날짜 컬럼을 datetime으로 변환
df['strt_dt_dt'] = pd.to_datetime(df['strt_dt_dt'])

# 연도, 월, 일, 주차, 시간 추출
df['year'] = df['strt_dt_dt'].dt.year
df['month'] = df['strt_dt_dt'].dt.month
df['day'] = df['strt_dt_dt'].dt.day
df['weekday'] = df['strt_dt_dt'].dt.weekday  # 0~6
df['hour'] = df['strt_dt_dt'].dt.hour

# sin-cos 주기적 변환 함수
def encode_cyclic(df, col, max_val):
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

# 월 (1~12), 일 (1~31), 요일 (0~6), 시간 (0~23)
df = encode_cyclic(df, 'month', 12)
df = encode_cyclic(df, 'day', 31)
df = encode_cyclic(df, 'weekday', 7)
df = encode_cyclic(df, 'hour', 24)

# ALS

In [8]:
from scipy import sparse

USER_COL  = "user_index"   # 필요 시 수정
ITEM_COL  = "asset_id"
RATING_COL = "view_ratio"   # 0~5 범위라 가정

# 범주형 → 순차 인덱스
user_codes, user_uniques = pd.factorize(df[USER_COL])
item_codes, item_uniques = pd.factorize(df[ITEM_COL])

alpha = 40.0                      # 명시적 평점을 implicit confidence 로 키우는 계수
data  = (df[RATING_COL] * alpha).astype(np.float32)

interaction_coo = sparse.coo_matrix(
    (data, (user_codes, item_codes)),
    shape=(len(user_uniques), len(item_uniques)),
    dtype=np.float32
)

print(f"희소 행렬 크기: {interaction_coo.shape}")
print(f"데이터 밀도: {interaction_coo.nnz / (interaction_coo.shape[0]*interaction_coo.shape[1]):.6f}")

희소 행렬 크기: (621976, 262728)
데이터 밀도: 0.000105


In [9]:
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm

# implicit 패키지는 (item × user) CSR 형태를 기대
item_user_csr = interaction_coo.T.tocsr()

als = AlternatingLeastSquares(
    factors=64,
    regularization=0.015,
    iterations=20,
    calculate_training_loss=True,
    use_gpu=False          # CUDA 가능 환경이면 True
)

# 진행률 표시용 함수 덮어쓰기 (선택)
als.progress = lambda x, **k: tqdm(total=x)  # tqdm 적용

als.fit(item_user_csr)
print("✅ 학습 완료")

c:\Users\user\anaconda3\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

✅ 학습 완료


In [10]:
# --- 학습 완료 후 ---
if hasattr(als, "progress"):
    del als.progress        # picklable 하지 않은 속성 삭제

In [12]:
import numpy as np
import pickle
import pathlib
from scipy import sparse

# 저장 디렉토리 설정
SAVE_DIR = pathlib.Path("artifacts")
SAVE_DIR.mkdir(exist_ok=True)

# 3) 요청한 형태로 각 구성요소 별도 저장 ────────────────────────────

# ALS 모델만 별도로 저장
with open(SAVE_DIR / "als.pkl", "wb") as f:
    pickle.dump(als, f)

# 사용자-아이템 CSR 행렬 (학습에 사용한 행렬)
user_item_csr = interaction_coo.tocsr()  # (user * item) CSR 형태로 변환
sparse.save_npz(SAVE_DIR / "user_item.npz", user_item_csr)

# 외부 사용자 ID -> 내부 인덱스 매핑 딕션너리
user_id_map = {user_id: idx for idx, user_id in enumerate(user_uniques)}
with open(SAVE_DIR / "user_map.pkl", "wb") as f:
    pickle.dump(user_id_map, f)

# 외부 아이템 ID -> 내부 인덱스 매핑 딕션너리
item_id_map = {item_id: idx for idx, item_id in enumerate(item_uniques)}
with open(SAVE_DIR / "item_map.pkl", "wb") as f:
    pickle.dump(item_id_map, f)

print("✅ 요청한 형태로 모델 및 관련 정보 저장 완료")
print(f"  - als.pkl: AlternatingLeastSquares 모델 객체")
print(f"  - user_item.npz: 훈련에 사용한 (user * item) CSR 행렬")
print(f"  - user_map.pkl: 외부 사용자 ID -> 내부 인덱스 매핑")
print(f"  - item_map.pkl: 외부 아이템 ID -> 내부 인덱스 매핑")

✅ 요청한 형태로 모델 및 관련 정보 저장 완료
  - als.pkl: AlternatingLeastSquares 모델 객체
  - user_item.npz: 훈련에 사용한 (user * item) CSR 행렬
  - user_map.pkl: 외부 사용자 ID -> 내부 인덱스 매핑
  - item_map.pkl: 외부 아이템 ID -> 내부 인덱스 매핑


# 컨텐츠 기반 필터링 아이템 프로필(Okt + TF-IDF)

In [13]:
TEXT_COLS = {
    "super_asset_nm": "[TITLE]",   # 제목
    "genre":          "[GENRE]",   # 장르(복수 가능)
    "category_l2":    "[CAT]",     # 카테고리
    "smry":           "[PLOT]",    # 줄거리/요약
}

NUM_COLS   = [                                # 수치형
    "month_sin", "month_cos",
    "day_sin", "day_cos",
    "weekday_sin", "weekday_cos",
    "hour_sin", "hour_cos",
    'rating_qt'
]

In [14]:
# 2. 텍스트 컬럼 합치기
# ------------------------------------------------------------
def concat_text(row):
    parts = []
    for col, prefix in TEXT_COLS.items():
        txt = str(row[col]) if pd.notna(row[col]) else ""
        if txt:
            parts.append(f"{prefix} {txt}")
    return " ".join(parts)

corpus = df.apply(concat_text, axis=1)

In [ ]:
from mecab import MeCab
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. MeCab 토크나이저 및 불용어 정의
# ------------------------------------------------------------
mecab = MeCab()

# 남기고 싶은 품사(NNG: 일반명사, NNP: 고유명사, VV: 동사, VA: 형용사, XR: 어근)
TAG_KEEP_MECAB = {'NNG', 'NNP', 'VV', 'VA', 'XR'}

# 일반 한국어 불용어
korean_stopwords = [
    '것', '수', '이', '그', '데', '저', '또', '를', '에', '의', '가', '을', '는', '들', '로', '으로', '써',
    '에서', '에게', '부터', '까지', '이런', '저런', '그런', '때', '곳', '년', '월', '일', '씨', '등', '중', 
    '같은', '같이', '요', '및', '즉', '또한', '때문', '뿐', '따라', '그리고', '하지만', '대한', '만큼',
    '자', '면', '시', '몇', '후', '각', '달', '적', '당', '타', '말', '직', '과', '와', '되다', '하다'
]

# VOD 도메인 특화 불용어
domain_stopwords = [
    '영화', '드라마', '방송', '시청', '보기', '컨텐츠', '장면', '화면', '작품', '감독', '배우',
    '시리즈', '에피소드', '시즌', '스트리밍', '채널', '보다', '관람', '제작', '출연'
]

# 최종 불용어 집합
all_stopwords = set(korean_stopwords + domain_stopwords)

def mecab_tokenizer(text: str):
    """
    1) MeCab으로 형태소 분석
    2) 지정한 품사만 유지
    3) 불용어 제거
    4) 토큰 리스트 반환
    """
    tokens = [
        morph for morph, pos in mecab.pos(text)
        if pos in TAG_KEEP_MECAB
    ]
    tokens = [token for token in tokens if token not in all_stopwords]
    return tokens

In [ ]:
corpus_list = corpus.tolist()

tokenized_docs = list(
    tqdm(                          # tqdm 이 자동으로 update(1) 호출
        map(mecab_tokenizer, corpus_list),
        total=len(corpus_list),
        desc="MeCab 토큰화 진행"
    )
)

# 3-2. 이미 토큰화된 문서를 그대로 사용하기 위해 토크나이저를 우회
class IdentityTokenizer:
    def __call__(self, doc):
        return doc

tfidf = TfidfVectorizer(
    tokenizer=IdentityTokenizer(),  # 토큰을 그대로 사용
    lowercase=False,                # 이미 소문자/대문자 처리 완료
    ngram_range=(1, 2),
    max_features=20_000,
    min_df=2,
    max_df=0.8
)

print("TF-IDF 벡터화 시작…")
X_text = tfidf.fit_transform(tokenized_docs)
print("완료! 행렬 크기:", X_text.shape)

MeCab 토큰화 진행:   0%|          | 0/17167525 [00:00<?, ?it/s]

In [ ]:

# (선택) 필드 가중치
weights = { "[TITLE]": 2.0, "[GENRE]": 3.0, "[CAT]": 2.5, "[PLOT]": 1.0 }
vocab, scale = tfidf.vocabulary_, np.ones(X_text.shape[1])
for tok, w in weights.items():
    key = tok.lower()
    if key in vocab:
        scale[vocab[key]] = w
X_text = X_text.multiply(scale)

In [ ]:
# 단어 빈도 확인 - 상위 30개 단어
from collections import Counter
import pandas as pd

# 토큰화된 결과 모음
all_tokens = []
for text in corpus:
    all_tokens.extend(mecab_tokenizer(text))

# 빈도수 계산
word_counts = Counter(all_tokens)

# 상위 30개 단어
top_words = pd.DataFrame(word_counts.most_common(30), columns=['단어', '빈도수'])
print("불용어 처리 후 가장 많이 등장한 단어:")
top_words